In [31]:
import os
from pathlib import Path
import json
import pandas as pd
from mri_data import utils
import nibabel as nib
from statistics import mean
import pyperclip
import sys
import numpy as np
from tqdm.notebook import tqdm
sys.path.append("/home/srs-9/Projects/prl_project")
from helpers.shell_interface import command, run_if_missing, open_itksnap_workspace_cmd


In [22]:
dataroot = Path("/media/smbshare/srs-9/prl_project/data")
inf_root = Path("/media/smbshare/srs-9/prl_project/training/roi_train2/run2/ensemble_output")
subject_sessions = pd.read_csv("/home/srs-9/Projects/prl_project/resources/subject-sessions.csv", index_col="sub")

suffix = "_xy20_z2"

In [23]:
with open(f"datalist{suffix}.json", 'r') as f:
    datalist = json.load(f)
    
test_data = datalist["testing"]
prl_data = []
lesion_data = []
for scan in test_data:
    scan = {k: Path(v) if isinstance(v, str) and ".nii.gz" in v else v for k, v in scan.items()}
    if  "prl" in Path(scan['label']).name:
        scan['inference'] = inf_root / Path(scan['label']).relative_to(dataroot).with_name(f"flair.phase{suffix}_ensemble.nii.gz")
        prl_data.append(scan)
    else:
        scan['inference'] = inf_root / Path(scan['label']).relative_to(dataroot).with_name(f"flair.phase{suffix}_ensemble.nii.gz")
        lesion_data.append(scan)


In [24]:
prl_dice_scores = []
for scan in prl_data:
    lab_data = nib.load(scan['label']).get_fdata()
    inf_data = nib.load(scan['inference']).get_fdata()
    prl_dice_scores.append(utils.dice_score(lab_data, inf_data, seg1_val=2, seg2_val=2))

In [26]:
print(prl_dice_scores)
print(mean(prl_dice_scores))

[0.27232796486090777, 0.44676409185803756, 0.502970297029703, 0.6951048951048951, 0.5601436265709157, 0.562268803945746, 0.48138437336130047, 0.555, 0.35526315789473684]
0.4923585789584714


In [ ]:
def get_performance2(lab_path, inf_path):
    lab = nib.load(lab_path).get_fdata()
    inf = nib.load(inf_path).get_fdata()
    TP = np.sum((lab == 2) & (inf == 2))
    FP = np.sum((lab == 1) & (inf == 2))
    TN = np.sum((lab == 1) & (inf == 1))
    FN = np.sum((lab == 2) & (inf == 1))
    return TP, FP, TN, FN

In [36]:
performance = []
all_data = prl_data + lesion_data
for scan in tqdm(all_data, total=len(all_data)):
    lab_path = scan['label']
    inf_path = scan['inference']
    performance.append(get_performance2(lab_path, inf_path))

  0%|          | 0/197 [00:00<?, ?it/s]

In [37]:
TP_total = 0
FP_total = 0
TN_total = 0
FN_total = 0
for TP, FP, TN, FN in performance:
    TP_total += TP
    FP_total += FP
    TN_total += TN
    FN_total += FN
    
sens = TP_total / (TP_total + FN_total)
spec = TN_total / (TN_total + FP_total)
ppv = TP_total / (TP_total + FP_total)
npv = TN_total / (TN_total + FN_total)

print("Sensitivity: {:0.4f}".format(sens))
print("Specificity: {:0.4f}".format(spec))
print("PPV:         {:0.4f}".format(ppv))
print("NPV:         {:0.4f}".format(npv))

Sensitivity: 0.6917
Specificity: 0.9678
PPV:         0.4289
NPV:         0.9890


In [33]:
performance

[(0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0, 0),
 (0, 0, 0,

In [ ]:
# view_root = Path("Z:/")
view_root = Path("/media/smbshare")
for scan in prl_data:
    images = [scan['image'].with_name(im) for im in [f"flair{suffix}.nii.gz", f"phase{suffix}.nii.gz"]]
    images = [view_root/(im.relative_to("/media/smbshare")) for im in images]
    labels = [scan['label'], scan['inference']]
    labels = [view_root/(im.relative_to("/media/smbshare")) for im in labels]
    cmd = open_itksnap_workspace_cmd(images, labels=labels, rename_root=("/media/smbshare", "Z:/"))
    print(cmd)

In [14]:
lesion_dice_scores = []
for scan in lesion_data:
    lab_data = nib.load(scan['label']).get_fdata()
    inf_data = nib.load(scan['inference']).get_fdata()
    lesion_dice_scores.append(utils.dice_score(lab_data, inf_data, seg1_val=2, seg2_val=2))

In [15]:
check_lesions = []
for i, (scan, dice) in enumerate(zip(lesion_data, lesion_dice_scores)):
    if dice is not None:
        check_lesions.append(scan)

In [16]:
view_root = Path("Z:/")
for scan in check_lesions:
    images = [scan['image'].with_name(im) for im in ["flair.nii.gz", "phase.nii.gz"]]
    images = [view_root/(im.relative_to("/media/smbshare")) for im in images]
    labels = [scan['label'], scan['inference']]
    labels = [view_root/(im.relative_to("/media/smbshare")) for im in labels]
    cmd = utils.open_itksnap_workspace_cmd(images, labels=labels)
    print(cmd)

itksnap -g Z:/srs-9/prl_project/data/sub1011-20180911/3/flair.nii.gz -o Z:/srs-9/prl_project/data/sub1011-20180911/3/phase.nii.gz -s Z:/srs-9/prl_project/data/sub1011-20180911/3/lesion.nii.gz Z:/srs-9/prl_project/training/roi_train1_segresnet/ensemble_output/sub1011-20180911/3/flair.phase_ensemble.nii.gz
itksnap -g Z:/srs-9/prl_project/data/sub1038-20161031/2/flair.nii.gz -o Z:/srs-9/prl_project/data/sub1038-20161031/2/phase.nii.gz -s Z:/srs-9/prl_project/data/sub1038-20161031/2/lesion.nii.gz Z:/srs-9/prl_project/training/roi_train1_segresnet/ensemble_output/sub1038-20161031/2/flair.phase_ensemble.nii.gz
itksnap -g Z:/srs-9/prl_project/data/sub1348-20181009/2/flair.nii.gz -o Z:/srs-9/prl_project/data/sub1348-20181009/2/phase.nii.gz -s Z:/srs-9/prl_project/data/sub1348-20181009/2/lesion.nii.gz Z:/srs-9/prl_project/training/roi_train1_segresnet/ensemble_output/sub1348-20181009/2/flair.phase_ensemble.nii.gz
itksnap -g Z:/srs-9/prl_project/data/sub2131-20190117/2/flair.nii.gz -o Z:/srs-9/p

In [16]:
subid = 2026
sesid = subject_sessions.loc[subid, "ses"]
subject_root = dataroot / f"sub{subid}-{sesid}"
l = 2
images = [subject_root/f"{l}/phase.nii.gz", subject_root/f"{l}/flair.nii.gz"]
segs = [subject_root/f"{l}/prl_label_final.nii.gz"]
cmd = utils.open_itksnap_workspace_cmd(images, labels=segs)
print(cmd)
pyperclip.copy(cmd)

itksnap -g /media/smbshare/srs-9/prl_project/data/sub2026-20160917/2/phase.nii.gz -o /media/smbshare/srs-9/prl_project/data/sub2026-20160917/2/flair.nii.gz -s /media/smbshare/srs-9/prl_project/data/sub2026-20160917/2/prl_label_final.nii.gz


In [19]:
segmentations = [
    "prl_mask_def_prob_CH.nii.gz",
    "prl_mask_def_prob_SRS.nii.gz",
    "prl_mask_def_prob_LR.nii.gz",
]
subid = 2026
sesid = subject_sessions.loc[subid, "ses"]
subject_root = dataroot / f"sub{subid}-{sesid}"

for seg in segmentations:
    if (subject_root / seg).exists():
        break
images = [subject_root/"phase.nii.gz", subject_root/"flair.nii.gz"]
segs = [subject_root/seg, subject_root/"lstai_lesion_index.nii.gz"]
cmd = utils.open_itksnap_workspace_cmd(images, labels=segs)
print(cmd)
pyperclip.copy(cmd)

itksnap -g /media/smbshare/srs-9/prl_project/data/sub2026-20160917/phase.nii.gz -o /media/smbshare/srs-9/prl_project/data/sub2026-20160917/flair.nii.gz -s /media/smbshare/srs-9/prl_project/data/sub2026-20160917/prl_mask_def_prob_SRS.nii.gz /media/smbshare/srs-9/prl_project/data/sub2026-20160917/lstai_lesion_index.nii.gz
